In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

# Create an API client
client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [2]:
# Add a user turn to the running conversation list.
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


# Add an assistant turn to preserve conversation history.
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


# Send the full message history to Claude and return plain text.
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Start with an empty conversation
messages = []

# Ask the model to generate an EventBridge rule as JSON
add_user_message(messages, "Generate a very short EventBridge rule as JSON")

# Prefill the assistant's response with the opening code fence to force JSON output
add_assistant_message(messages, "```json")

# Stop generation at the closing code fence so we only capture the JSON body
text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "MyRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [4]:
import json

json.loads(text.strip())

{'Name': 'MyRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

In [5]:
## Exercise

# Start with an empty conversation
messages = []

# User prompt asking for three short AWS CLI commands
prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

# Add the prompt as a user message
add_user_message(messages, prompt)

# Prefill the assistant's response to force a single bash code block with no comments
add_assistant_message(
    messages,
    "Here are all three commands in a single block without any comments:\n ```bash",
)

# Stop generation at the closing code fence so we only capture the commands
text = chat(messages, stop_sequences=["```"])
# text = chat(messages)
text

'\naws s3 ls\naws ec2 describe-instances\naws iam list-users\n '

In [6]:
from IPython.display import Markdown

Markdown(text)


aws s3 ls
aws ec2 describe-instances
aws iam list-users
 